In [ ]:
# CELL 1 — Install
# Install only approved libraries for this sprint.
!pip install "transformers>=4.40.0" "peft>=0.10.0" "trl>=0.8.6" "bitsandbytes>=0.43.0" "datasets>=2.18.0" "accelerate>=0.29.0" wandb openai jsonlines python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.9 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
!git clone https://github.com/awbasit/claracare-diabetes.git

Cloning into 'claracare-diabetes'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 50 (delta 4), reused 50 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 5.24 MiB | 11.13 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [ ]:
%cd /content/claracare-diabetes
!ls

/content/claracare-diabetes
data  notebooks  README.md  requirements.txt  run.py  src  util


In [ ]:
# CELL 2 — Auth + Environment Check
import os
import sys
import re
from datetime import datetime
from pathlib import Path
from getpass import getpass
from google.colab import userdata

import matplotlib.pyplot as plt

import torch
import wandb
from huggingface_hub import login
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

# Make repo utilities importable in Colab, even if repo folder name changes.
sys.path.append("/content/claracare-diabetes")
sys.path.append("/content/claracare-diabetes/util")

from util.prompting import build_prompt
from util.data_ops import load_jsonl

PROJECT_NAME = 'claracare-checkpoints'
RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
LOG_TO_WANDB = True

WANDB_API_KEY = userdata.get('WANDB_API_KEY')
HF_TOKEN = userdata.get('HF_TOKEN')
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

login(HF_TOKEN, add_to_git_credential=True)
# Log in to Weights & Biases
wandb_api_key = userdata.get('WANDB_API_KEY')
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

# Configure Weights & Biases to record against our project
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"

if LOG_TO_WANDB:
  wandb.init(project=PROJECT_NAME, name=RUN_NAME)

assert torch.cuda.is_available(), 'CUDA GPU not available.'
print('GPU:', torch.cuda.get_device_name(0))
print('Total VRAM (GB):', torch.cuda.get_device_properties(0).total_memory / 1e9)

DRIVE = Path('/content/drive/MyDrive/claracare-checkpoints')
TRAIN_PATH = Path('/content/claracare-diabetes/data/processed/claracare_train.jsonl')
EVAL_PATH = Path('/content/claracare-diabetes/data/processed/claracare_eval.jsonl')
SFT_CKPT = DRIVE / 'sft-adapter'
SFT_MERGED = DRIVE / 'sft-merged'
DPO_CKPT = DRIVE / 'dpo-adapter'
DPO_MERGED = DRIVE / 'dpo-merged'
PREF_PATH = DRIVE / 'preference_pairs.jsonl'

DRIVE.mkdir(parents=True, exist_ok=True)

if not TRAIN_PATH.exists() or not EVAL_PATH.exists():
    print('Missing required files in /content.')
    print('Upload claracare_train.jsonl and claracare_eval.jsonl into Colab session storage.')
    raise AssertionError('Dataset files missing.')

print('Environment check passed.')

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: nanaqwadjobarima (nanaqwadjobarima-university-of-mines-and-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


GPU: Tesla T4
Total VRAM (GB): 15.637086208
Environment check passed.


In [ ]:
# CELL 3 — Load Eval Set + Run BASELINE Inference
from util.data_ops import load_jsonl, save_jsonl
from util.model_ops import make_4bit_config, load_tokenizer, load_model_4bit, run_eval_single

BASE_MODEL = 'mistralai/Mistral-7B-Instruct-v0.3'
BASELINE_OUT = DRIVE / 'baseline_outputs.jsonl'

eval_rows = load_jsonl(EVAL_PATH)
assert len(eval_rows) == 158, f'Expected 158 eval rows, got {len(eval_rows)}'

bnb_config = make_4bit_config()
tokenizer = load_tokenizer(BASE_MODEL)
model = load_model_4bit(BASE_MODEL, bnb_config)

baseline_rows = run_eval_single(eval_rows, model, tokenizer, temperature=0.3)
save_jsonl(BASELINE_OUT, baseline_rows)

for i in range(3):
    print(f'\n--- Baseline sample {i+1} ---')
    print('Instruction:', baseline_rows[i]['instruction'])
    print('Model:', baseline_rows[i]['model_output'][:500])
    print(f"Processed {i}/{len(eval_rows)}")

assert BASELINE_OUT.exists(), 'Baseline file not saved.'
print('Baseline saved.')

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]


--- Baseline sample 1 ---
Instruction: What causes weakness in muscles and difficulty in holding objects?

i feel like my muscle are getting weak, sometime when am holding on to something it like i just want to drop it. and my legs are getting like a charlie horse in my calves and my feet and toes cramp up too. plus my middle back it tight too at time what cause these things???
Model: I'm really sorry to hear you're experiencing these symptoms. Weak muscles, hand tremors, and cramps in your legs, feet, and toes could be signs of high blood sugar levels, which is a common symptom of Type 2 diabetes. Additionally, back pain can also be a symptom.

It's essential to speak with your doctor about these symptoms, as they can provide a proper diagnosis and recommend the best course of action. If it is diabetes, managing your blood sugar levels can help alleviate these symptoms.

In 
Processed 0/158

--- Baseline sample 2 ---
Instruction: Can diabetes be reversed using diet,exercise and herba

In [ ]:
# CELL 4 — QLoRA + BitsAndBytes Config (Define Only)
import torch
from peft import LoraConfig
from transformers import BitsAndBytesConfig, TrainingArguments

sft_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

sft_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    task_type='CAUSAL_LM',
)

sft_args_dict = {
    'output_dir': str(SFT_CKPT),
    'num_train_epochs': 3,
    'per_device_train_batch_size': 2,
    'gradient_accumulation_steps': 8,
    'learning_rate': 2e-4,
    'lr_scheduler_type': 'cosine',
    'warmup_ratio': 0.03,
    'max_seq_length': 1024,
    'fp16': True,
    'gradient_checkpointing': True,
    'save_steps': 50,
    'save_total_limit': 3,
    'logging_steps': 10,
    'report_to': 'wandb',
    'run_name': 'claracare-sft',
}

sft_train_args = TrainingArguments(**{k: v for k, v in sft_args_dict.items() if k != 'max_seq_length'})
sft_max_seq_length = sft_args_dict['max_seq_length']

print('BitsAndBytesConfig:', sft_bnb_config)
print('LoraConfig:', sft_lora_config)
print('SFT args:')
for k, v in sft_args_dict.items():
    print(f'  {k}: {v}')

In [ ]:
# CELL 5 — Load Model + Apply SFT LoRA
from peft import get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM
from util.model_ops import load_tokenizer

BASE_MODEL = 'mistralai/Mistral-7B-Instruct-v0.3'

tokenizer = load_tokenizer(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=sft_bnb_config, device_map='auto')
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, sft_lora_config)
model.print_trainable_parameters()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
pct = 100.0 * trainable / total
print(f'Trainable% = {pct:.4f}')
assert 0.0 < pct <= 5.0, f'Invalid trainable ratio: {pct:.4f}%'
assert pct < 2.1, 'Trainable ratio too high for expected LoRA setup.'

In [ ]:
# CELL 6 — Format SFT Dataset
from datasets import load_dataset
from util.prompting import format_sft

train_ds = load_dataset('json', data_files=str(TRAIN_PATH), split='train')
assert len(train_ds) == 3001, f'Expected 3001 rows, got {len(train_ds)}'

formatted_preview = [format_sft(train_ds[i]) for i in range(2)]
for i, text in enumerate(formatted_preview, 1):
    print(f'\n--- Formatted sample {i} ---\n{text[:900]}')

assert all(x.strip() for x in formatted_preview), 'Empty formatted sample found.'
print('SFT dataset format check passed.')

In [ ]:
# CELL 7 — SFT Training
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=sft_train_args,
    train_dataset=train_ds,
    formatting_func=format_sft,
    max_seq_length=sft_max_seq_length,
    tokenizer=tokenizer,
)
trainer.train()

In [ ]:
# CELL 8 — Save SFT Adapter
import os

trainer.model.save_pretrained(str(SFT_CKPT))
tokenizer.save_pretrained(str(SFT_CKPT))

print('SFT adapter files:')
print(os.listdir(SFT_CKPT))

In [ ]:
# CELL 9 — Merge SFT Adapter Into Base Model
import os
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

print(torch.cuda.memory_allocated() / 1e9, 'GB used before merge load')

base_fp16 = AutoModelForCausalLM.from_pretrained(
    'mistralai/Mistral-7B-Instruct-v0.3',
    torch_dtype=torch.float16,
    device_map='auto',
)
merge_tok = AutoTokenizer.from_pretrained('mistralai/Mistral-7B-Instruct-v0.3', use_fast=True)

merged = PeftModel.from_pretrained(base_fp16, str(SFT_CKPT))
merged = merged.merge_and_unload()

SFT_MERGED.mkdir(parents=True, exist_ok=True)
merged.save_pretrained(str(SFT_MERGED))
merge_tok.save_pretrained(str(SFT_MERGED))

print('SFT merged saved:', SFT_MERGED)
print('Files:', os.listdir(SFT_MERGED))

del merged
del base_fp16
torch.cuda.empty_cache()

In [ ]:
# CELL 10 — Run SFT Inference on Eval Set (Paired)
from util.data_ops import load_jsonl, save_jsonl
from util.model_ops import make_4bit_config, load_tokenizer, load_model_4bit, run_eval_paired

PAIRED_OUT = DRIVE / 'sft_outputs_paired.jsonl'

eval_rows = load_jsonl(EVAL_PATH)
assert len(eval_rows) == 157, f'Expected 157 eval rows, got {len(eval_rows)}'

bnb_inf = make_4bit_config()
tokenizer = load_tokenizer(str(SFT_MERGED))
model = load_model_4bit(str(SFT_MERGED), bnb_inf)

paired_rows = run_eval_paired(eval_rows, model, tokenizer, cold=0.3, warm=0.9)
save_jsonl(PAIRED_OUT, paired_rows)

for i in range(2):
    print(f'\n--- Paired sample {i+1} ---')
    print(paired_rows[i])

assert PAIRED_OUT.exists(), 'Paired output file missing.'
print('Saved:', PAIRED_OUT)

In [ ]:
# CELL 11 — Generate Preference Pairs (GPT-4o-mini Judge)
import json
import os
import time
from openai import OpenAI

from util.data_ops import load_jsonl, save_jsonl
from util.prompting import build_prompt

assert os.environ.get('OPENAI_API_KEY'), 'OPENAI_API_KEY missing.'
client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])

paired_path = DRIVE / 'sft_outputs_paired.jsonl'
paired_rows = load_jsonl(paired_path)
assert len(paired_rows) == 157, f'Expected 157 paired rows, got {len(paired_rows)}'

SYSTEM_JUDGE = (
    "You are evaluating responses from a health literacy assistant\n"
    "for Type 2 diabetes patients in Ghana with low health literacy.\n"
    "Given a patient question and two responses, choose which response is\n"
    "better for this specific population.\n"
    "Criteria (in order of importance):\n"
    "1. Simplicity — grade 6-7 reading level, short sentences, no jargon\n"
    "2. Safety — no specific dosages, recommends doctor for personal decisions\n"
    "3. Accuracy — medically correct for Type 2 diabetes\n"
    "4. Ghana context — relevant to West African patient experience\n"
    "Return ONLY valid JSON. No preamble. No markdown.\n"
    "Schema: {\n"
    "  chosen: 'A' or 'B',\n"
    "  score_a: float 1.0-5.0,\n"
    "  score_b: float 1.0-5.0,\n"
    "  reason: str (one sentence max)\n"
    "}"
)

def judge_pair(row: dict) -> dict:
    """Judge one A/B response pair with GPT-4o-mini."""
    user_msg = (
        f"Question: {row['instruction']}\n"
        f"Response A: {row['model_output_a']}\n"
        f"Response B: {row['model_output_b']}\n"
        "Which response better serves a low-literacy diabetes patient in Ghana?"
    )
    res = client.chat.completions.create(
        model='gpt-4o-mini',
        response_format={'type': 'json_object'},
        messages=[
            {'role': 'system', 'content': SYSTEM_JUDGE},
            {'role': 'user', 'content': user_msg},
        ],
        temperature=0.0,
    )
    return json.loads(res.choices[0].message.content)

kept, failures = [], []
threshold = 1.0

for i, row in enumerate(paired_rows, 1):
    try:
        judgment = judge_pair(row)
        chosen = judgment['chosen']
        score_a = float(judgment['score_a'])
        score_b = float(judgment['score_b'])

        if abs(score_a - score_b) >= threshold:
            if chosen == 'A':
                chosen_text, rejected_text = row['model_output_a'], row['model_output_b']
            elif chosen == 'B':
                chosen_text, rejected_text = row['model_output_b'], row['model_output_a']
            else:
                raise AssertionError(f'Invalid chosen value: {chosen}')

            kept.append(
                {
                    'prompt': build_prompt(row['instruction']),
                    'chosen': chosen_text,
                    'rejected': rejected_text,
                }
            )
    except Exception as exc:
        failures.append({'index': i, 'error': str(exc)})

    if i % 10 == 0:
        save_jsonl(PREF_PATH, kept)
        print(f'Processed {i}/157, kept {len(kept)}')

    time.sleep(0.5)

save_jsonl(PREF_PATH, kept)
print(f'kept {len(kept)}/157 high-confidence pairs')
print(f'failures: {len(failures)}')
assert len(kept) >= 60, 'Too few pairs (<60). Lower threshold to 0.75 and regenerate.'

In [ ]:
# CELL 12 — DPO Training Config (Define Only)
from peft import LoraConfig
from trl import DPOConfig

dpo_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    task_type='CAUSAL_LM',
)

dpo_config = DPOConfig(
    output_dir=str(DPO_CKPT),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    fp16=True,
    gradient_checkpointing=True,
    beta=0.1,
    save_steps=20,
    save_total_limit=2,
    logging_steps=5,
    report_to='wandb',
    run_name='claracare-dpo',
    max_length=1024,
    max_prompt_length=512,
)

print('DPO LoraConfig:', dpo_lora_config)
print('DPOConfig:', dpo_config)

In [ ]:
# CELL 13 — Load SFT Model for DPO + Apply DPO LoRA
from datasets import load_dataset
from peft import get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM
from util.model_ops import make_4bit_config, load_tokenizer

bnb_dpo = make_4bit_config()
tokenizer = load_tokenizer(str(SFT_MERGED))

model = AutoModelForCausalLM.from_pretrained(
    str(SFT_MERGED),
    quantization_config=bnb_dpo,
    device_map='auto',
)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, dpo_lora_config)
model.print_trainable_parameters()

preference_dataset = load_dataset('json', data_files=str(PREF_PATH), split='train')
for col in ['prompt', 'chosen', 'rejected']:
    assert col in preference_dataset.column_names, f'Missing column: {col}'
assert len(preference_dataset) >= 60, 'Too few preference pairs for DPO'
print('Preference rows:', len(preference_dataset))

In [ ]:
# CELL 14 — DPO Training
from trl import DPOTrainer

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=preference_dataset,
    tokenizer=tokenizer,
    beta=0.1,
)
trainer.train()

In [ ]:
# CELL 15 — Save DPO Adapter + Merge
import os
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

trainer.model.save_pretrained(str(DPO_CKPT))
tokenizer.save_pretrained(str(DPO_CKPT))
print('DPO adapter files:', os.listdir(DPO_CKPT))

base_sft_fp16 = AutoModelForCausalLM.from_pretrained(
    str(SFT_MERGED),
    torch_dtype=torch.float16,
    device_map='auto',
)
merge_tok = AutoTokenizer.from_pretrained(str(SFT_MERGED), use_fast=True)

dpo_merged = PeftModel.from_pretrained(base_sft_fp16, str(DPO_CKPT))
dpo_merged = dpo_merged.merge_and_unload()

DPO_MERGED.mkdir(parents=True, exist_ok=True)
dpo_merged.save_pretrained(str(DPO_MERGED))
merge_tok.save_pretrained(str(DPO_MERGED))

print('DPO merged saved:', DPO_MERGED)
print('Files:', os.listdir(DPO_MERGED))

del dpo_merged
del base_sft_fp16
torch.cuda.empty_cache()

In [ ]:
# CELL 16 — Final Inference: DPO Model on Eval Set
from util.data_ops import load_jsonl, save_jsonl
from util.model_ops import make_4bit_config, load_tokenizer, load_model_4bit, run_eval_single

DPO_OUT = DRIVE / 'dpo_outputs.jsonl'
PAIRED_OUT = DRIVE / 'sft_outputs_paired.jsonl'
BASELINE_OUT = DRIVE / 'baseline_outputs.jsonl'

eval_rows = load_jsonl(EVAL_PATH)

bnb_inf = make_4bit_config()
tokenizer = load_tokenizer(str(DPO_MERGED))
model = load_model_4bit(str(DPO_MERGED), bnb_inf)

dpo_rows = run_eval_single(eval_rows, model, tokenizer, temperature=0.3)
save_jsonl(DPO_OUT, dpo_rows)
assert DPO_OUT.exists(), 'DPO output file missing.'

baseline_rows = load_jsonl(BASELINE_OUT)
paired_rows = load_jsonl(PAIRED_OUT)
baseline_map = {x['instruction']: x['model_output'] for x in baseline_rows}
sft_map = {x['instruction']: x['model_output_a'] for x in paired_rows}

for i in range(3):
    q = dpo_rows[i]['instruction']
    print('\n==============================')
    print('Question:', q)
    print('Baseline:', baseline_map.get(q, '')[:500])
    print('SFT:     ', sft_map.get(q, '')[:500])
    print('DPO:     ', dpo_rows[i]['model_output'][:500])

## OOM Recovery

### T4 OOM during SFT
1. `max_seq_length` 1024 -> 512
2. `per_device_train_batch_size` 2 -> 1 and `gradient_accumulation_steps` 8 -> 16
3. Restart runtime and resume from latest checkpoint in Drive

### T4 OOM during DPO merge (Cell 15)
1. Restart runtime to clear VRAM fully
2. Run only merge cells
3. Load `SFT_MERGED` from Drive (already saved)

### T4 OOM during DPO training (Cell 14)
1. Keep `per_device_train_batch_size=1`
2. Increase `gradient_accumulation_steps` 8 -> 16
3. Reduce `max_length` 1024 -> 512 in `DPOConfig`

In [ ]:
# CELL 17 — Sprint 2 Deliverables Validation
from util.validation import validate_sprint2_outputs

BASELINE_OUT = DRIVE / 'baseline_outputs.jsonl'
PAIRED_OUT = DRIVE / 'sft_outputs_paired.jsonl'
DPO_OUT = DRIVE / 'dpo_outputs.jsonl'

validate_sprint2_outputs(
    baseline_out=BASELINE_OUT,
    sft_ckpt=SFT_CKPT,
    sft_merged=SFT_MERGED,
    paired_out=PAIRED_OUT,
    pref_path=PREF_PATH,
    dpo_ckpt=DPO_CKPT,
    dpo_merged=DPO_MERGED,
    dpo_out=DPO_OUT,
)
print('All Sprint 2 deliverable checks passed.')